In [1]:
import pandas as pd
import numpy as np

### Step 1. Load the data

In [2]:
df = pd.read_csv('tech_layoffs.csv')
df

,company,total_layoffs,impacted_workforce_percentage,reported_date,industry,headquarter_location,sources,status,additional_notes
0,C2FO,20,2,12/9/2022,"Fintech, payments","Leawood, KS",Kansas City Business Journal,Private,NaN
1,VideoAmp,Unclear,2,9/19/2022,Advertising platforms,Los Angeles,Company memo,Private,NaN
2,Amperity,13,3,8/20/2022,"Saas, analytics",Seattle,Geekwire,Private,NaN
3,Addepar,20,3,1/18/2023,"fintech, data analytics","Mountain View, CA",Fortune,Private,NaN
4,Thirty Madison,24,3,5/13/2022,Health Care,New York,Axios,Private,"Per Axios, 22 of the employees laid off were f..."
...,...,...,...,...,...,...,...,...,...
484,Salesforce,9090,Unclear,1/4/2023,"enterprise tech, analytics",San Francisco,New York Times,Public,NaN
485,Oracle,Unclear,Unclear,8/1/2022,"SaaS, enterprise software",Austin,The Information,Public,The Information reported thousands of people w...
486,LivePerson,Unclear,Unclear,9/15/2022,Customer service,New York,LinkedIn,Public,NaN
487,Skillz,Unclear,Unclear,8/29/2022,"Mobile Apps, Gaming",San Francisco,Seeking Alpha,Public,NaN


### Step 2. What is wrong, and how much ?

In [3]:
summary = (pd.DataFrame({
    'dtypes':df.dtypes,
    'missing_values': df.isna().sum(),
    'missing_%': (df.isna().mean()*100).round(1),
    'unique': df.nunique()
}))
print(summary)
print('Exact duplicate rows:', df.duplicated().sum())

                              dtypes  missing_values  missing_%  unique
company                          str               0        0.0     477
total_layoffs                    str               0        0.0     150
impacted_workforce_percentage    str               0        0.0      50
reported_date                    str               0        0.0     187
industry                         str               0        0.0     289
headquarter_location             str               0        0.0     134
sources                          str               0        0.0     221
status                           str               0        0.0       2
additional_notes                 str             467       95.5      22
Exact duplicate rows: 0


### Step 3. Column names

In [4]:
df.columns = (df.columns.str.strip()
                        .str.lower()
                        .str.replace(r'\s+','_',regex=True))
df.columns.tolist()

['company',
 'total_layoffs',
 'impacted_workforce_percentage',
 'reported_date',
 'industry',
 'headquarter_location',
 'sources',
 'status',
 'additional_notes']

In [5]:
df = df.drop(columns=['additional_notes']) # becuase we do not need this column for our analysis.

### Step 4. Normalize text

In [6]:
text_cols = ['company','industry','headquarter_location','sources','status']
for col in text_cols:
    df[col] = (df[col].str.strip()
                      .str.replace(r'\s+',' ',regex=True)
                      .str.lower())
df[['company','industry','headquarter_location','sources','status']].head()

,company,industry,headquarter_location,sources,status
0,c2fo,"fintech, payments","leawood, ks",kansas city business journal,private
1,videoamp,advertising platforms,los angeles,company memo,private
2,amperity,"saas, analytics",seattle,geekwire,private
3,addepar,"fintech, data analytics","mountain view, ca",fortune,private
4,thirty madison,health care,new york,axios,private


### Step 5. Handling sentinel values

In [7]:
df = df.replace('Unclear',np.nan) # missing values are represented by 'Unclear' in whole dataset.
df

,company,total_layoffs,impacted_workforce_percentage,reported_date,industry,headquarter_location,sources,status
0,c2fo,20,2,12/9/2022,"fintech, payments","leawood, ks",kansas city business journal,private
1,videoamp,NaN,2,9/19/2022,advertising platforms,los angeles,company memo,private
2,amperity,13,3,8/20/2022,"saas, analytics",seattle,geekwire,private
3,addepar,20,3,1/18/2023,"fintech, data analytics","mountain view, ca",fortune,private
4,thirty madison,24,3,5/13/2022,health care,new york,axios,private
...,...,...,...,...,...,...,...,...
484,salesforce,9090,NaN,1/4/2023,"enterprise tech, analytics",san francisco,new york times,public
485,oracle,NaN,NaN,8/1/2022,"saas, enterprise software",austin,the information,public
486,liveperson,NaN,NaN,9/15/2022,customer service,new york,linkedin,public
487,skillz,NaN,NaN,8/29/2022,"mobile apps, gaming",san francisco,seeking alpha,public


### Step 6. Removing duplicates

In [8]:
df = df.drop_duplicates(subset=['company','total_layoffs','reported_date','industry'],keep='last')
df = df.reset_index(drop=True)

### Step 7. Text to numbers

In [9]:
df['total_layoffs'] = pd.to_numeric(df['total_layoffs'],errors='coerce')
df['impacted_workforce_percentage'] = pd.to_numeric(df['impacted_workforce_percentage'],errors='coerce')
df[['total_layoffs','impacted_workforce_percentage']].dtypes

total_layoffs                    float64
impacted_workforce_percentage    float64
dtype: object

### Step 8. Splitting fields (Is one column hiding several?)

In [10]:
parts = df['headquarter_location'].str.extract(r'^(?P<city>[^,]+?)(?:,\s*(?P<region>.+))?$')
df['hq_city'] = parts['city'].str.title()
df['hq_region'] = parts['region'].str.upper()
df = df.drop(columns=['headquarter_location'])
df[['hq_city','hq_region']].head()

,hq_city,hq_region
0,Leawood,KS
1,Los Angeles,NaN
2,Seattle,NaN
3,Mountain View,CA
4,New York,NaN


### Step 9. Standardize Categories

In [11]:
city_fix = {
    'San Franicsco': 'San Francisco',
    'San Francsico': 'San Francisco',
    'Dever': 'Denver', 
    'New York City': 'New York',
    'Greater New York Area': 'New York', 
    'Playa Visa': 'Playa Vista',
    'Distributed': np.nan,
}
region_fix = {
    'CALIFORNIA': 'CA', 
    'NEW JERSEY': 'NJ', 
    'OREGON': 'OR', 
    'UTAH': 'UT',
    'MASSACHUSETTS': 'MA', 
    'NORTH CAROLINA': 'NC', 
    'VIRGINIA': 'VA',
}
industry_fix = {
    'cybsersecurity': 'cybersecurity',
    'transporation':  'transportation',
    'ecommerce':      'e-commerce',
    'foodtech':       'food tech',
    'virtual events': 'virtual event',
}
df['hq_city'] = df['hq_city'].replace(city_fix)
df['hq_region'] = df['hq_region'].replace(region_fix)
for wrong , right in industry_fix.items():
    df['industry'] = df['industry'].str.replace(wrong, right, regex=False)
df['main_industry'] = df['industry'].str.split(',').str[0].str.strip()
group = {
    'crypto':               'cryptocurrency',
    'blockchain':           'cryptocurrency',
    'ai':                   'artificial intelligence',
    'marijuana':            'cannabis',
    'gaming':               'video games',
    'esports':              'video games',
    'esports/video games':  'video games',
    'security':             'cybersecurity',
    'shipping':             'logistics',
    'shipping and logistics': 'logistics',
    'productivity tools':   'productivity',
    'productivity software': 'productivity',
    'enterprise tech':      'enterprise software',
}
df['main_industry'] = df['main_industry'].replace(group)
df = df.drop(columns=['industry'])
df['main_industry']

0                    fintech
1      advertising platforms
2                       saas
3                    fintech
4                health care
               ...          
484      enterprise software
485                     saas
486         customer service
487              mobile apps
488      enterprise software
Name: main_industry, Length: 489, dtype: object

### Step 10. Validating formats

In [12]:
pattern = r'^\d{1,2}/\d{1,2}/\d{4}$'
valid = df['reported_date'].astype(str).str.match(pattern)
print('Dates not in M/D/YYYY format:', (~valid).sum())

Dates not in M/D/YYYY format: 0


### Step 11. Domain Rules

In [13]:
Rules = {
    'total_layoffs': (0,50000),
    'impacted_workforce_percentage':(0,100)
}
for col, (low,high) in Rules.items():
    bad = df[col].notna() & ~df[col].between(low,high)
    print(f'{col}: {bad.sum()} values(s) outside [{low},{high}]')
    df.loc[bad,col] = np.nan

total_layoffs: 0 values(s) outside [0,50000]
impacted_workforce_percentage: 0 values(s) outside [0,100]


### Step 12. Missing data

In [14]:
df.isna().sum()

company                            0
total_layoffs                    182
impacted_workforce_percentage    133
reported_date                      0
sources                            0
status                             0
hq_city                            1
hq_region                        367
main_industry                      0
dtype: int64

In [15]:
df['hq_city'] = df['hq_city'].fillna('Unknown')

| Column | Missing | Decision | Reason |
|---|---|---|---|
| `total_layoffs` | 182 | Leave as `NaN` | Companies did not report it, and filling would invent layoffs |
| `impacted_workforce_percentage` | 133 | Leave as `NaN` | Same reason |
| `hq_region` | 367 | Leave as `NaN` | Many cities simply have no region |
| `hq_city` | 1 | Fill with `'Unknown'` | Keeps the row without inventing a city |
| All other columns | 0 | Nothing to do | No missing values |

### Step 13. Outliers

In [16]:
q1 = df['total_layoffs'].quantile(0.25)
q3 = df['total_layoffs'].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
df['layoffs_outlier'] = (df['total_layoffs'] < low) | (df['total_layoffs'] > high)
print('Outliers flagged:', df['layoffs_outlier'].sum())
df.loc[df['layoffs_outlier'],['company','main_industry','reported_date','total_layoffs']]\
    .sort_values(by='total_layoffs',ascending=False).head(10)

Outliers flagged: 46


,company,main_industry,reported_date,total_layoffs
378,amazon,e-commerce,1/5/2023,18000.0
470,alphabet,big tech,1/20/2023,12000.0
420,meta,social media,11/9/2022,11000.0
469,microsoft,enterprise software,1/18/2023,10000.0
484,salesforce,enterprise software,1/4/2023,9090.0
224,better.com,fintech,4/19/2022,5000.0
381,cisco,enterprise software,11/16/2022,4100.0
475,peloton,fitness,10/6/2022,4084.0
466,carvana,automotive,1/13/2023,4000.0
463,twitter,social media,1/5/2023,3740.0


### Step 14. Data Types

In [17]:
df['total_layoffs'] = df['total_layoffs'].astype('Int64')
df['impacted_workforce_percentage'] = df['impacted_workforce_percentage'].astype('Int64')
df['reported_date'] = pd.to_datetime(df['reported_date'],format='%m/%d/%Y',errors='coerce')

for col in ['status','main_industry','hq_city','hq_region']:
    df[col] = df[col].astype('category')
df.dtypes

company                                     str
total_layoffs                             Int64
impacted_workforce_percentage             Int64
reported_date                    datetime64[us]
sources                                     str
status                                 category
hq_city                                category
hq_region                              category
main_industry                          category
layoffs_outlier                            bool
dtype: object

### Step 15. Feature Engineering

In [18]:
df['month'] = df['reported_date'].dt.to_period('M').astype('str')
df['layoff_size'] = pd.cut(df['total_layoffs'],[0,50,200,1000,np.inf],
                           labels=['Small','Medium','Large','Huge'],
                           include_lowest=True)
df['impact_level'] = pd.cut(df['impacted_workforce_percentage'],[0,5,15,100],
                            labels=['Low','Medium','High'],
                            include_lowest=True)
print(df['layoff_size'].value_counts())
print("-" * 30)
print(df['impact_level'].value_counts())

layoff_size
Medium    145
Small      89
Large      54
Huge       19
Name: count, dtype: int64
------------------------------
impact_level
High      172
Medium    147
Low        37
Name: count, dtype: int64


### Step 16. Encoding

In [19]:
top = df['main_industry'].value_counts().head(10).index
industry_small = df['main_industry'].astype(str).where(df['main_industry'].isin(top), 'other')

model_df = pd.concat([
    df[['total_layoffs', 'impacted_workforce_percentage']].astype('float64'),
    pd.get_dummies(df[['status']], drop_first=True, dtype=int),
    pd.get_dummies(industry_small, prefix='industry', drop_first=True, dtype=int),
], axis='columns')

model_df = model_df.dropna()      
print(model_df.shape)
model_df.head()

(240, 13)


,total_layoffs,impacted_workforce_percentage,status_public,industry_cryptocurrency,industry_cybersecurity,industry_e-commerce,industry_edtech,industry_enterprise software,industry_fintech,industry_health care,industry_insurtech,industry_other,industry_proptech
0,20.0,2.0,0,0,0,0,0,0,1,0,0,0,0
2,13.0,3.0,0,0,0,0,0,0,0,0,0,1,0
3,20.0,3.0,0,0,0,0,0,0,1,0,0,0,0
4,24.0,3.0,0,0,0,0,0,0,0,1,0,0,0
6,25.0,4.0,0,0,1,0,0,0,0,0,0,0,0


### Step 17. Splitting (Train / Test)

In [20]:
train = model_df.sample(frac=0.8, random_state=42)
test = model_df.drop(train.index)
print('train:', train.shape, '| test:', test.shape)

train: (192, 13) | test: (48, 13)


### Step 18. Validate and compare

In [21]:
df_raw = pd.read_csv("tech_layoffs.csv")   

print('BEFORE:', df_raw.shape)
print(pd.DataFrame({'dtype': df_raw.dtypes.astype(str),
                    'missing': df_raw.isna().sum(),
                    'unique': df_raw.nunique()}))
print("-" * 30)
print('AFTER:', df.shape)
print(pd.DataFrame({'dtype': df.dtypes.astype(str),
                    'missing': df.isna().sum(),
                    'unique': df.nunique()}))

BEFORE: (489, 9)
                              dtype  missing  unique
company                         str        0     477
total_layoffs                   str        0     150
impacted_workforce_percentage   str        0      50
reported_date                   str        0     187
industry                        str        0     289
headquarter_location            str        0     134
sources                         str        0     221
status                          str        0       2
additional_notes                str      467      22
------------------------------
AFTER: (489, 13)
                                        dtype  missing  unique
company                                   str        0     477
total_layoffs                           Int64      182     149
impacted_workforce_percentage           Int64      133      49
reported_date                  datetime64[us]        0     187
sources                                   str        0     218
status                     

In [22]:
essential = ['company', 'reported_date', 'main_industry', 'status']

assert not df.duplicated(subset=['company', 'reported_date', 'total_layoffs']).any(), 'duplicate layoff record remains'
assert df[essential].notna().all().all(), 'missing values in essential columns'
assert df['total_layoffs'].dropna().between(*Rules['total_layoffs']).all(), 'total_layoffs outside valid range'
assert df['impacted_workforce_percentage'].dropna().between(*Rules['impacted_workforce_percentage']).all(), 'percentage outside valid range'
assert df['reported_date'].between('2022-01-01', '2023-12-31').all(), 'date outside expected range'
assert not df['hq_city'].isin(['San Franicsco', 'San Francsico', 'Dever']).any(), 'city typo remains'
print('All checks passed.')

All checks passed.


In [23]:
df.to_csv('layoffs_clean_1.csv', index=False)